# 实验一 · AXPY 向量数乘累加  (y ← a·x + y)

**所属**：《并行计算》第三章 · ARM NEON SIMD 编程　|　**难度**：⭐ 入门　|　**预计时长**：20–30 分钟

> **实验说明**
> 1. 本实验采用分步实现的方式：由 **v1 串行版本**起，依次引入 **v2 NEON 向量化**与 **v3 循环展开**，每一版本均实际编译并运行，据此观察性能的逐步变化。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖 **ARM(aarch64/arm64) + NEON**；请在华为鲲鹏处理器上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. 三个版本的代码为递增关系：后一版本在前一版本基础上新增一个实现方法，输出表格相应增加一行，便于对照阅读，理解优化的引入过程。

## 🎯 学习目标

完成本实验后，学生应能够：

- 阐述 **SIMD（单指令多数据）** 的基本思想，理解一条指令并行处理 4 个单精度浮点数的机制
- 建立可靠的性能基准：通过 `no-tree-vectorize` 关闭自动向量化，获得未经向量化的参照实现
- 掌握**向量化三步曲**：加载 `vld1q` → 计算 `vfmaq` → 存储 `vst1q`
- 正确处理**尾部循环**，理解 `restrict` 关键字对编译器优化的作用
- 掌握**循环展开**技术，理解其提升指令级并行（ILP）的原理
- 在同一编译选项下比较**编译器自动向量化**与**手写 NEON 内联函数**，判断手写实现的适用场景
- 运用**算术强度**分析程序是否受限于访存带宽，解释理论加速比与实测加速比的差异

## 🗺️ 学习路径

1. **准备阶段**：理解 AXPY 的定义及其作为入门案例的原因；掌握 NEON 向量化三步曲
2. **v1 · 串行基准**：实现 `axpy_serial_no_vec`（关闭向量化，作为基准）与 `axpy_serial`（允许编译器自动向量化）
   → 考察编译器自动向量化所能达到的性能水平
3. **v2 · NEON 向量化**：新增 `axpy_neon`，比较三个版本的性能
   → 考察手写内联函数相对于编译器自动向量化的差异
4. **v3 · 循环展开**：新增 `axpy_neon_unroll`，完成四个版本的完整对比
   → 考察循环展开带来的额外性能收益
5. **可视化与分析**：以 v3 的输出结果绘制加速比柱状图，分析性能瓶颈，引出**访存墙**概念

## 1. 背景与动机

AXPY 是 BLAS **Level-1** 中最基础的操作：`y = a·x + y`（a 为标量，x、y 为向量）。

选择该操作作为入门案例的原因在于其**结构简单**——不含分支，元素之间无依赖关系，因此可将学习重点集中于 SIMD 机制本身：如何将标量循环改写为一次处理多个元素的向量循环。掌握本实验的方法后，后续较复杂的图像处理算法均可视为该机制的组合与扩展。

## 2. 算法与公式

对每个元素执行：

$$y_i \leftarrow a \cdot x_i + y_i$$

该运算为典型的 **FMA（乘加融合）**：一次乘法与一次加法。NEON 的 `float32x4_t` 类型为 128 位向量，可容纳 4 个单精度浮点数，配合 `vfmaq_f32` 即可用**一条指令完成 4 个元素**的 `a·x+y` 运算，理论吞吐量为标量实现的 4 倍。

> ⚠️ **注意**：AXPY 对 y **原地更新**。因此每次计时前必须将 y 复位为初始值（代码中的 `memcpy(y_opt, y_init, bytes)`），且复位操作不计入计时区间。

## 3. 核心 NEON 指令与技巧

```c
float32x4_t va = vdupq_n_f32(a);            // 将标量 a 复制到 4 条通道
for (i = 0; i + 4 <= n; i += 4) {
  float32x4_t vx = vld1q_f32(x + i);        // ① 加载 4 个 x
  float32x4_t vy = vld1q_f32(y + i);        //   加载 4 个 y
  vy = vfmaq_f32(vy, vx, va);               // ② FMA：vy = vx*va + vy
  vst1q_f32(y + i, vy);                     // ③ 存回 4 个 y
}
// 尾部：当 n 不是 4 的整数倍时，剩余元素以标量方式处理
```

- **`vld1q_f32` / `vst1q_f32`**：连续加载 / 存储 4 个单精度浮点数
- **`vfmaq_f32(a, b, c)`**：计算 `a + b*c`，乘加运算一步完成
- **`vdupq_n_f32(a)`**：将标量复制到全部 4 条通道
- **`restrict`**：声明 x 与 y 所指内存不重叠，帮助编译器判断指针之间不存在别名（aliasing），从而进行更激进的优化
- **循环展开**：单次迭代处理 8 或 16 个元素，以减少循环控制开销并提高访存并行度

## 4. 环境准备

In [ ]:
import platform, subprocess, shutil, sys

print("Python  :", sys.version.split()[0])
print("架构    :", platform.machine())
CC = shutil.which("gcc") or shutil.which("clang") or shutil.which("cc")
print("编译器  :", CC)
IS_ARM = platform.machine().lower() in ("aarch64", "arm64", "armv7l", "armv8l")
if not IS_ARM:
    print("\n⚠️  当前不是 ARM 架构，NEON 代码无法在此编译运行。")
elif CC is None:
    print(
        "\n⚠️  未找到 C 编译器，请先安装 gcc（如 sudo apt install build-essential）。"
    )
else:
    print("\n✅ 环境就绪：ARM 架构 + 编译器可用，可以开始实验！")

In [ ]:
import subprocess, platform, re, shutil

MACHINE = platform.machine().lower()


def compile_c(src, out):
    """尝试多组编译参数，返回可执行文件名；失败则打印错误。"""
    base = shutil.which("gcc") or shutil.which("cc") or "cc"
    if MACHINE in ("armv7l", "armv8l"):  # 32 位 ARM 需显式开 NEON
        flagsets = ["-O3 -fPIC -mfpu=neon -mfloat-abi=hard -march=armv7-a"]
    else:  # aarch64 / arm64：NEON 默认开启
        flagsets = ["-O3 -fPIC"]
    last = ""
    for fl in flagsets:
        cmd = f"{base} {fl} {src} -o {out} -lm"
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            print("✅ 编译成功：", cmd)
            return out
        last = r.stderr
    print("❌ 编译失败：\n", last)
    return None


def run_bin(out, *args):
    """运行可执行文件并打印其输出。"""
    r = subprocess.run(
        [f"./{out}"] + [str(a) for a in args], capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print("STDERR:", r.stderr)
    return r.stdout


def parse_table(text):
    """解析 | 方法 | 耗时 | 加速比 | 校验 | 表格，兼容 '4.21 x' 与 '4.21x'。"""
    rows = []
    for line in text.splitlines():
        s = line.strip()
        if not s.startswith("|"):
            continue
        cells = [c.strip() for c in s.strip("|").split("|")]
        if len(cells) < 3:
            continue
        name = cells[0]
        if name.lower() in ("method", "方法") or set(name) <= set("-: "):
            continue
        mt = re.search(r"[-+]?\d*\.?\d+", cells[1])
        ms = re.search(r"[-+]?\d*\.?\d+", cells[2])
        if not mt:
            continue
        rows.append(
            {
                "method": name,
                "time": float(mt.group()),
                "speedup": float(ms.group()) if ms else None,
            }
        )
    return rows

In [ ]:
import matplotlib.pyplot as plt


def plot_speedup(rows, title=""):
    rows = [r for r in rows if r["speedup"] is not None]
    if not rows:
        print("未解析到可绘制的加速比。")
        return
    names = [r["method"] for r in rows]
    sp = [r["speedup"] for r in rows]
    best = sp.index(max(sp))
    colors = ["#9aa0a6" if s <= 1.05 else "#295E96" for s in sp]
    colors[best] = "#C7000B"  # 最快版本标红
    plt.figure(figsize=(8, 4))
    bars = plt.bar(names, sp, color=colors)
    plt.axhline(1.0, ls="--", c="gray", lw=1)
    for b, s in zip(bars, sp):
        plt.text(
            b.get_x() + b.get_width() / 2,
            s,
            f"{s:.2f}x",
            ha="center",
            va="bottom",
            fontsize=10,
        )
    plt.ylabel("Speedup (x)")
    plt.title(title)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# 创建源代码目录
!mkdir -p src_axpy

## 5. v1 · 串行基准实现

第一个版本包含**两个函数体完全相同**的串行实现，其区别仅在于是否允许编译器进行自动向量化：

<table>
<thead>
<tr>
<th style="text-align:left">函数</th>
<th style="text-align:left">说明</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align:left"><code>axpy_serial_no_vec</code></td>
<td style="text-align:left">通过 <code>__attribute__((optimize("no-tree-vectorize")))</code> <strong>显式关闭</strong>自动向量化。该版本作为本实验的<strong>性能基准</strong>，其余各版本的加速比均以其为 1.00× 参照</td>
</tr>
<tr>
<td style="text-align:left"><code>axpy_serial</code></td>
<td style="text-align:left">源码相同，但<strong>允许编译器自动向量化</strong></td>
</tr>
</tbody>
</table>

### 💡 为何需要两个串行版本
若直接以普通串行代码作为基准，编译器可能已对其自动向量化，导致所测得的加速比被低估，无法准确反映 SIMD 带来的性能提升。因此必须显式关闭向量化，以获得未经优化的参照实现。

而 `axpy_serial` 的引入，则用于回答另一个问题：**在相同编译选项下，编译器自动向量化能达到何种性能水平**。

### 代码要点
- `check_diff()`：将结果与参考值比对，容差为 `1e-5`，返回 `PASS` 或 `FAIL`
- `aligned_alloc(16, bytes)`：按 16 字节对齐分配内存，以利于 SIMD 访存
- `y_init` 保存 y 的初始值；每次计时前通过 `memcpy` 复位，复位操作不计入计时
- 计时循环执行 `NTIMES` 次取平均值；程序结束前调用 `free()` 释放内存

运行后，请观察 `Serial (Auto)` 相对基准的加速比。

In [ ]:
%%writefile src_axpy/axpy_v1.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Helper: Get monotonic time in MILLISECONDS (ms)
double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference
const char* check_diff(const float* y_ref, const float* y_test, long n) {
  double max_diff = 0.0;
  for (long i = 0; i < n; i++) {
    double diff = fabs((double)y_ref[i] - (double)y_test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  if (max_diff < 1e-5)
    return "PASS";
  else
    return "FAIL";
}

// ---------------------------------------------------------
// 1. Serial Version (Forced NO Vectorization) -- BASELINE
// This version serves as the performance baseline; the speedup of all
// other versions is measured relative to it.
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void axpy_serial_no_vec(const float* restrict x, float* restrict y, float a,
                        long n) {
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

// ---------------------------------------------------------
// 2. Serial Version (Auto-vectorization friendly)
// Identical source code, but the compiler is free to vectorize it.
// ---------------------------------------------------------
void axpy_serial(const float* restrict x, float* restrict y, float a, long n) {
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    printf("Usage: %s <vector_length>\n", argv[0]);
    return 1;
  }

  long n = atol(argv[1]);

  printf("=======================================================\n");
  printf(
      " AXPY v1: Serial Baseline and Auto-Vectorized Serial (BLAS Level-1)\n");
  printf(" Vector Length: %ld\n", n);
  printf(" Loops:         %d\n", NTIMES);
  printf("=======================================================\n");

  // Aligned allocation for SIMD efficiency (16-byte alignment)
  size_t bytes = sizeof(float) * n;
  bytes = (bytes + 15) & ~((size_t)15);
  float* x = (float*)aligned_alloc(16, bytes);
  float* y_init = (float*)aligned_alloc(16, bytes);  // Pristine initial Y
  float* y_ref = (float*)aligned_alloc(16, bytes);   // Reference result
  float* y_opt = (float*)aligned_alloc(16, bytes);   // Working buffer

  if (!x || !y_init || !y_ref || !y_opt) {
    printf("Memory allocation failed\n");
    return 1;
  }

  // Initialization. NOTE: AXPY updates y in place, so y must be reset
  // to y_init before every fresh application.
  for (long i = 0; i < n; i++) {
    x[i] = i * 0.1f;
    y_init[i] = (float)(i % 13) * 0.5f;
  }
  float a = 2.5f;

  // Reference result = ONE application of the scalar kernel
  memcpy(y_ref, y_init, bytes);
  axpy_serial_no_vec(x, y_ref, a, n);

  double start, end;

  // 1. Baseline: Serial (No Vectorization)
  printf("Running Serial (No-Vec)...\n");
  memcpy(y_opt, y_init, bytes);  // reset before timing (reset not timed)
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_serial_no_vec(x, y_opt, a, n);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;

  // 2. Serial (Auto-Vectorization)
  printf("Running Serial (Auto-Vec)...\n");
  memcpy(y_opt, y_init, bytes);
  axpy_serial(x, y_opt, a, n);  // single clean application for correctness
  const char* s_serial = check_diff(y_ref, y_opt, n);
  memcpy(y_opt, y_init, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_serial(x, y_opt, a, n);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;

  // Report
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_no_vec);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);
  printf("-------------------------------------------------\n");

  free(x);
  free(y_init);
  free(y_ref);
  free(y_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_axpy/axpy_v1.c", "src_axpy/axpy_v1")
out_v1 = run_bin(BIN, 10500000)

## 6. v2 · NEON 内联函数实现

在 v1 的基础上**新增 `axpy_neon` 函数**，采用向量化三步曲改写循环：

```c
float32x4_t va = vdupq_n_f32(a);      // 标量 a 复制到 4 条通道
for (; i <= n - 4; i += 4) {
  float32x4_t vx = vld1q_f32(x + i);  // ① 加载
  float32x4_t vy = vld1q_f32(y + i);
  vy = vfmaq_f32(vy, vx, va);         // ② 乘加
  vst1q_f32(y + i, vy);               // ③ 存储
}
for (; i < n; i++) y[i] = a * x[i] + y[i];   // 尾部以标量处理
```

### 🔑 知识点
- **单条指令处理 4 个单精度浮点数**：`float32x4_t` 为 128 位向量类型，可容纳 4 个 32 位浮点数
- **尾循环的必要性**：`n` 未必为 4 的整数倍，剩余的 1–3 个元素须以标量方式处理
- **`vfmaq_f32` 为融合乘加指令**：乘法与加法在单条指令内完成，较之分别使用 `vmulq` 与 `vaddq`，具有更高的执行效率与更好的数值精度（❓思考下为什么）

此时程序包含**三个版本**。NEON 实现的理论加速比为 4×，请观察其实测值，并与 `Serial (Auto)` 的结果进行比较。

In [ ]:
%%writefile src_axpy/axpy_v2.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Helper: Get monotonic time in MILLISECONDS (ms)
double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference
const char* check_diff(const float* y_ref, const float* y_test, long n) {
  double max_diff = 0.0;
  for (long i = 0; i < n; i++) {
    double diff = fabs((double)y_ref[i] - (double)y_test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  if (max_diff < 1e-5)
    return "PASS";
  else
    return "FAIL";
}

// ---------------------------------------------------------
// 1. Serial Version (Forced NO Vectorization) -- BASELINE
// This version serves as the performance baseline; the speedup of all
// other versions is measured relative to it.
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void axpy_serial_no_vec(const float* restrict x, float* restrict y, float a,
                        long n) {
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

// ---------------------------------------------------------
// 2. Serial Version (Auto-vectorization friendly)
// Identical source code, but the compiler is free to vectorize it.
// ---------------------------------------------------------
void axpy_serial(const float* restrict x, float* restrict y, float a, long n) {
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

// ---------------------------------------------------------
// 3. NEON Intrinsic Version (Basic)
// Vectorization in 3 steps: Load -> Compute (FMA) -> Store
// ---------------------------------------------------------
void axpy_neon(const float* restrict x, float* restrict y, float a, long n) {
  float32x4_t va = vdupq_n_f32(a);  // Load 'a' into all 4 lanes

  long i = 0;
  for (; i <= n - 4; i += 4) {
    float32x4_t vx = vld1q_f32(x + i);  // Load 4 floats from x
    float32x4_t vy = vld1q_f32(y + i);  // Load 4 floats from y (accumulator)
    vy = vfmaq_f32(vy, vx, va);  // Fused Multiply-Add: vy = vy + (vx * va)
    vst1q_f32(y + i, vy);        // Store 4 floats back to y
  }

  // Tail loop: Handle remaining elements
  for (; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    printf("Usage: %s <vector_length>\n", argv[0]);
    return 1;
  }

  long n = atol(argv[1]);

  printf("=======================================================\n");
  printf(" AXPY v2: Add NEON Intrinsic Implementation (BLAS Level-1)\n");
  printf(" Vector Length: %ld\n", n);
  printf(" Loops:         %d\n", NTIMES);
  printf("=======================================================\n");

  // Aligned allocation for SIMD efficiency (16-byte alignment)
  size_t bytes = sizeof(float) * n;
  bytes = (bytes + 15) & ~((size_t)15);
  float* x = (float*)aligned_alloc(16, bytes);
  float* y_init = (float*)aligned_alloc(16, bytes);  // Pristine initial Y
  float* y_ref = (float*)aligned_alloc(16, bytes);   // Reference result
  float* y_opt = (float*)aligned_alloc(16, bytes);   // Working buffer

  if (!x || !y_init || !y_ref || !y_opt) {
    printf("Memory allocation failed\n");
    return 1;
  }

  // Initialization. NOTE: AXPY updates y in place, so y must be reset
  // to y_init before every fresh application.
  for (long i = 0; i < n; i++) {
    x[i] = i * 0.1f;
    y_init[i] = (float)(i % 13) * 0.5f;
  }
  float a = 2.5f;

  // Reference result = ONE application of the scalar kernel
  memcpy(y_ref, y_init, bytes);
  axpy_serial_no_vec(x, y_ref, a, n);

  double start, end;

  // 1. Baseline: Serial (No Vectorization)
  printf("Running Serial (No-Vec)...\n");
  memcpy(y_opt, y_init, bytes);  // reset before timing (reset not timed)
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_serial_no_vec(x, y_opt, a, n);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;

  // 2. Serial (Auto-Vectorization)
  printf("Running Serial (Auto-Vec)...\n");
  memcpy(y_opt, y_init, bytes);
  axpy_serial(x, y_opt, a, n);  // single clean application for correctness
  const char* s_serial = check_diff(y_ref, y_opt, n);
  memcpy(y_opt, y_init, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_serial(x, y_opt, a, n);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;

  // 3. NEON Intrinsic (Basic)
  printf("Running NEON Basic...\n");
  memcpy(y_opt, y_init, bytes);
  axpy_neon(x, y_opt, a, n);
  const char* s_neon = check_diff(y_ref, y_opt, n);
  memcpy(y_opt, y_init, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_neon(x, y_opt, a, n);
  end = get_time_ms();
  double t_neon = (end - start) / NTIMES;

  // Report
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_no_vec);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_neon,
         t_no_vec / t_neon, s_neon);
  printf("-------------------------------------------------\n");

  free(x);
  free(y_init);
  free(y_ref);
  free(y_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_axpy/axpy_v2.c", "src_axpy/axpy_v2")
out_v2 = run_bin(BIN, 10500000)

## 7. v3 · 循环展开实现

在 v2 的基础上**新增 `axpy_neon_unroll` 函数**：单次迭代处理 **16 个元素**（即 4 个向量）。

```c
for (; i <= n - 16; i += 16) {
  float32x4_t vx0 = vld1q_f32(x + i);      // 4 组相互独立的加载
  float32x4_t vx1 = vld1q_f32(x + i + 4);
  ...
  vy0 = vfmaq_f32(vy0, vx0, va);           // 4 条互不依赖的 FMA 指令
  vy1 = vfmaq_f32(vy1, vx1, va);
  ...
}
// 其后处理剩余的 4 元素组，最后以标量方式处理尾部
```

### 🔑 知识点：循环展开的性能收益来源
1. **降低循环控制开销**：循环计数、边界判断与跳转指令的执行次数减少至原先的 1/4
2. **提高指令级并行（ILP）**：4 条 `vfmaq` 指令**互不依赖**，处理器可在流水线中重叠执行，无需等待前一条指令的结果
3. **提高访存并行度**：多条加载指令可同时处于执行状态，有助于隐藏内存访问延迟

> 需要注意的是，此处的展开使用了**多个相互独立的向量变量**（vy0–vy3），而非简单地复制循环体。

至此，**四个版本**全部实现完毕。

In [ ]:
%%writefile src_axpy/axpy_v3.c
#include <arm_neon.h>
#include <math.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define NTIMES 20

// Helper: Get monotonic time in MILLISECONDS (ms)
double get_time_ms() {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

// Helper: Verify results against reference
const char* check_diff(const float* y_ref, const float* y_test, long n) {
  double max_diff = 0.0;
  for (long i = 0; i < n; i++) {
    double diff = fabs((double)y_ref[i] - (double)y_test[i]);
    if (diff > max_diff) max_diff = diff;
  }
  if (max_diff < 1e-5)
    return "PASS";
  else
    return "FAIL";
}

// ---------------------------------------------------------
// 1. Serial Version (Forced NO Vectorization) -- BASELINE
// This version serves as the performance baseline; the speedup of all
// other versions is measured relative to it.
// ---------------------------------------------------------
#if defined(__GNUC__)
__attribute__((optimize("no-tree-vectorize")))
#endif
void axpy_serial_no_vec(const float* restrict x, float* restrict y, float a,
                        long n) {
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

// ---------------------------------------------------------
// 2. Serial Version (Auto-vectorization friendly)
// Identical source code, but the compiler is free to vectorize it.
// ---------------------------------------------------------
void axpy_serial(const float* restrict x, float* restrict y, float a, long n) {
  for (long i = 0; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

// ---------------------------------------------------------
// 3. NEON Intrinsic Version (Basic)
// Vectorization in 3 steps: Load -> Compute (FMA) -> Store
// ---------------------------------------------------------
void axpy_neon(const float* restrict x, float* restrict y, float a, long n) {
  float32x4_t va = vdupq_n_f32(a);  // Load 'a' into all 4 lanes

  long i = 0;
  for (; i <= n - 4; i += 4) {
    float32x4_t vx = vld1q_f32(x + i);  // Load 4 floats from x
    float32x4_t vy = vld1q_f32(y + i);  // Load 4 floats from y (accumulator)
    vy = vfmaq_f32(vy, vx, va);  // Fused Multiply-Add: vy = vy + (vx * va)
    vst1q_f32(y + i, vy);        // Store 4 floats back to y
  }

  // Tail loop: Handle remaining elements
  for (; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

// ---------------------------------------------------------
// 4. NEON Intrinsic Version (Loop Unrolling 4x)
// Process 16 elements per iteration to increase instruction-level
// parallelism (ILP) and to amortize the loop overhead.
// ---------------------------------------------------------
void axpy_neon_unroll(const float* restrict x, float* restrict y, float a,
                      long n) {
  float32x4_t va = vdupq_n_f32(a);

  long i = 0;
  for (; i <= n - 16; i += 16) {
    // Load x (multiple instruction issue opportunities)
    float32x4_t vx0 = vld1q_f32(x + i);
    float32x4_t vx1 = vld1q_f32(x + i + 4);
    float32x4_t vx2 = vld1q_f32(x + i + 8);
    float32x4_t vx3 = vld1q_f32(x + i + 12);

    // Load y (the accumulator)
    float32x4_t vy0 = vld1q_f32(y + i);
    float32x4_t vy1 = vld1q_f32(y + i + 4);
    float32x4_t vy2 = vld1q_f32(y + i + 8);
    float32x4_t vy3 = vld1q_f32(y + i + 12);

    // Computation: vy = vy + vx * va
    vy0 = vfmaq_f32(vy0, vx0, va);
    vy1 = vfmaq_f32(vy1, vx1, va);
    vy2 = vfmaq_f32(vy2, vx2, va);
    vy3 = vfmaq_f32(vy3, vx3, va);

    // Store
    vst1q_f32(y + i, vy0);
    vst1q_f32(y + i + 4, vy1);
    vst1q_f32(y + i + 8, vy2);
    vst1q_f32(y + i + 12, vy3);
  }

  // Handle remaining chunks of 4
  for (; i <= n - 4; i += 4) {
    float32x4_t vx = vld1q_f32(x + i);
    float32x4_t vy = vld1q_f32(y + i);
    vy = vfmaq_f32(vy, vx, va);
    vst1q_f32(y + i, vy);
  }

  // Handle remaining scalars
  for (; i < n; i++) {
    y[i] = a * x[i] + y[i];
  }
}

int main(int argc, char* argv[]) {
  if (argc != 2) {
    printf("Usage: %s <vector_length>\n", argv[0]);
    return 1;
  }

  long n = atol(argv[1]);

  printf("=======================================================\n");
  printf(" AXPY v3: Add NEON Loop-Unrolled Implementation (BLAS Level-1)\n");
  printf(" Vector Length: %ld\n", n);
  printf(" Loops:         %d\n", NTIMES);
  printf("=======================================================\n");

  // Aligned allocation for SIMD efficiency (16-byte alignment)
  size_t bytes = sizeof(float) * n;
  bytes = (bytes + 15) & ~((size_t)15);
  float* x = (float*)aligned_alloc(16, bytes);
  float* y_init = (float*)aligned_alloc(16, bytes);  // Pristine initial Y
  float* y_ref = (float*)aligned_alloc(16, bytes);   // Reference result
  float* y_opt = (float*)aligned_alloc(16, bytes);   // Working buffer

  if (!x || !y_init || !y_ref || !y_opt) {
    printf("Memory allocation failed\n");
    return 1;
  }

  // Initialization. NOTE: AXPY updates y in place, so y must be reset
  // to y_init before every fresh application.
  for (long i = 0; i < n; i++) {
    x[i] = i * 0.1f;
    y_init[i] = (float)(i % 13) * 0.5f;
  }
  float a = 2.5f;

  // Reference result = ONE application of the scalar kernel
  memcpy(y_ref, y_init, bytes);
  axpy_serial_no_vec(x, y_ref, a, n);

  double start, end;

  // 1. Baseline: Serial (No Vectorization)
  printf("Running Serial (No-Vec)...\n");
  memcpy(y_opt, y_init, bytes);  // reset before timing (reset not timed)
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_serial_no_vec(x, y_opt, a, n);
  end = get_time_ms();
  double t_no_vec = (end - start) / NTIMES;

  // 2. Serial (Auto-Vectorization)
  printf("Running Serial (Auto-Vec)...\n");
  memcpy(y_opt, y_init, bytes);
  axpy_serial(x, y_opt, a, n);  // single clean application for correctness
  const char* s_serial = check_diff(y_ref, y_opt, n);
  memcpy(y_opt, y_init, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_serial(x, y_opt, a, n);
  end = get_time_ms();
  double t_serial = (end - start) / NTIMES;

  // 3. NEON Intrinsic (Basic)
  printf("Running NEON Basic...\n");
  memcpy(y_opt, y_init, bytes);
  axpy_neon(x, y_opt, a, n);
  const char* s_neon = check_diff(y_ref, y_opt, n);
  memcpy(y_opt, y_init, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_neon(x, y_opt, a, n);
  end = get_time_ms();
  double t_neon = (end - start) / NTIMES;

  // 4. NEON Intrinsic (Loop Unrolling)
  printf("Running NEON Unrolled...\n");
  memcpy(y_opt, y_init, bytes);
  axpy_neon_unroll(x, y_opt, a, n);
  const char* s_unroll = check_diff(y_ref, y_opt, n);
  memcpy(y_opt, y_init, bytes);
  start = get_time_ms();
  for (int t = 0; t < NTIMES; t++) axpy_neon_unroll(x, y_opt, a, n);
  end = get_time_ms();
  double t_unroll = (end - start) / NTIMES;

  // Report
  printf("\n-------------------------------------------------\n");
  printf("| Method          | Time (ms) | Speedup | Check |\n");
  printf("|-----------------|-----------|---------|-------|\n");
  printf("| Serial (No-Vec) | %9.3f |  1.00 x |   -   |\n", t_no_vec);
  printf("| Serial (Auto)   | %9.3f | %5.2f x |  %-4s |\n", t_serial,
         t_no_vec / t_serial, s_serial);
  printf("| NEON Intrinsic  | %9.3f | %5.2f x |  %-4s |\n", t_neon,
         t_no_vec / t_neon, s_neon);
  printf("| NEON Unrolled   | %9.3f | %5.2f x |  %-4s |\n", t_unroll,
         t_no_vec / t_unroll, s_unroll);
  printf("-------------------------------------------------\n");

  free(x);
  free(y_init);
  free(y_ref);
  free(y_opt);
  return 0;
}

In [ ]:
BIN = compile_c("src_axpy/axpy_v3.c", "src_axpy/axpy_v3")
out_v3 = run_bin(BIN, 10500000)

## 8. 📈 性能可视化（基于 v3 的四版本结果）

v3 的输出包含全部四个版本的耗时与加速比，据此绘制柱状图，以完整呈现各优化手段的效果。
（灰色表示无明显加速，蓝色表示存在加速，**红色标示性能最优的版本**）

> 注：具体数值随硬件平台、编译器版本与系统负载而变化，请以本机实际运行结果为准，后续实验结果分析仅针对数据反映的**趋势与规律**。

In [ ]:
rows_v3 = parse_table(out_v3)
for r in rows_v3:
    print(f'{r["method"]:16s} {r["time"]:9.3f} ms   {r["speedup"]:.2f}x')
plot_speedup(rows_v3, "AXPY: performance of four implementations (N=10.5M)")

## 9. 结果分析

实验结果通常呈现以下三个现象：

**① NEON 实现相对基准有明显加速，但显著低于理论值 4×**。

其根本原因在于 **AXPY 属于访存受限型运算**：每处理一个元素需读取 x、读取 y 并写回 y（约 12 字节的访存量），而仅执行 2 次浮点运算。

> **算术强度** ≈ 2 FLOP / 12 Byte ≈ **0.17 FLOP/Byte**。该值过低，导致计算单元大部分时间处于等待内存数据的状态，向量宽度的增加无法转化为等比例的性能提升。这正是第一章 **Roofline 模型**中“访存受限区”的典型表现。

**② `Serial (Auto)` 的性能通常接近甚至等同于手写 NEON 实现。**

AXPY 属于**规整的逐元素循环**，不含规约运算与复杂的数据重排，编译器有能力对其实施有效的自动向量化。

**③ 三种向量化实现（Serial Auto、NEON Intrinsic、NEON Unrolled）的性能基本相当。**

三者的加速比十分接近，彼此差异处于测量误差范围之内，并无某一版本明显占优。这一现象恰恰印证了 AXPY 的**访存受限**本质：既然性能瓶颈在于内存带宽而非计算，那么无论采用编译器自动向量化、手写 NEON，还是进一步循环展开，都只是在改善“计算侧”，而计算侧本就不是瓶颈，因此各种手段殊途同归，加速比都被同一道“访存墙”限制在相近的水平。

---

### 🎓 结论
对于**规整、逐元素、不含规约**的循环，启用 `-O3` 使编译器自动向量化通常已可获得良好性能。**手写 NEON 内联函数的价值主要体现在编译器难以自动优化的场景**——例如规约运算、复杂的数据重排与饱和运算。

## 10. 🔧 动手练习

请修改代码、重新编译并运行，观察性能的变化（建议先独立完成，再阅读思考题）：

1. 将向量长度由 `10500000` 分别改为 `100000`（可驻留于缓存）与 `50000000`（远超缓存容量），对比加速比的变化并分析原因。
2. 将 v3 源码中的 `float` 全部改为 `double`（相应地 `float32x4_t`→`float64x2_t`、`vld1q_f32`→`vld1q_f64` 等）。一个向量仅能容纳 2 个双精度数，观察加速比的变化。
3. 将 v3 的展开因子由 16 改为 8 或 32，观察性能是否进一步提升，并确定合适的展开程度。
4. 【进阶】为 `compile_c` 的编译参数添加 `-march=native` 后重新运行 v3，观察 `Serial (Auto)` 的性能是否超过手写 NEON 实现，并解释其原因。

## 11. 🤔 思考题

- AXPY 的理论加速比为 4×，实测值为何低于该值？其瓶颈在于计算还是访存？
- 若某处理器的内存带宽提升一倍，AXPY 的加速比是否会改变？若计算单元数量提升一倍呢？
- 为何基准版本必须使用 `no-tree-vectorize` 显式关闭自动向量化？若不关闭会产生何种影响？
- `Serial (Auto)` 已获得相当的加速，在何种情形下才有必要手写 NEON 内联函数？
- 计时前的 `memcpy(y_opt, y_init, bytes)` 为何不能置于计时区间之内？

## 12. 小结与后续

本实验完成了“**串行 → 向量化 → 循环展开**”的完整优化过程：

<table>
<thead>
<tr>
<th style="text-align:left">版本</th>
<th style="text-align:left">新增内容</th>
<th style="text-align:left">涉及知识点</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align:left"><strong>v1</strong></td>
<td style="text-align:left"><code>axpy_serial_no_vec</code> + <code>axpy_serial</code></td>
<td style="text-align:left">性能基准的建立、编译器自动向量化</td>
</tr>
<tr>
<td style="text-align:left"><strong>v2</strong></td>
<td style="text-align:left"><code>axpy_neon</code></td>
<td style="text-align:left">向量化三步曲、尾循环处理、FMA 指令</td>
</tr>
<tr>
<td style="text-align:left"><strong>v3</strong></td>
<td style="text-align:left"><code>axpy_neon_unroll</code></td>
<td style="text-align:left">循环展开、指令级并行（ILP）</td>
</tr>
</tbody>
</table>

通过 AXPY，我们认识了 SIMD 的**基本机制**（向量化三步曲）及其**性能上限**（访存墙）。需要强调的是：**向量化并非普适的加速手段，其收益上限由算术强度与内存带宽共同决定**。

➡️ **后续内容：实验二 矩阵-向量乘（GEMV）**。当运算中出现求和操作时，将遇到 SIMD 的第一个实质性难点：**规约与数据依赖**。届时，手写 NEON 内联函数将具有明确的必要性。